# Lição 09 - Padrão de Projeto de Metacognição


## Configuração

Este notebook demonstra o padrão de design Metacognição usando o Microsoft Agent Framework.

**Pré-requisitos:**
- Implantação do Azure OpenAI configurada através de variáveis de ambiente
- Azure CLI autenticada (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## O que é Metacognição?

Metacognição é **pensar sobre o pensamento**. No contexto de agentes de IA, significa construir agentes que podem:

- **Refletir sobre si próprios** relativamente às suas próprias saídas e processo de raciocínio
- **Detetar erros** e recuperar de forma elegante em vez de falhar silenciosamente
- **Avaliar** se as suas respostas são completas e úteis
- **Adaptar** a sua estratégia quando uma abordagem inicial não funciona (p.ex., recorrer a um sistema de reserva)

Um agente metacognitivo não se limita a responder a perguntas — monitora o seu próprio desempenho e ajusta-se em tempo real.


## Ferramentas Primárias e de Reserva

Um padrão comum de metacognição é a **estratégia de fallback**. O agente tenta primeiro uma ferramenta primária; se esta falhar (por exemplo, um erro 404), o agente reconhece a falha e muda transparentemente para uma ferramenta de reserva.

Isto reflete sistemas do mundo real onde os serviços primários podem estar indisponíveis e os agentes devem autodiagnosticar o problema antes de escolher um caminho alternativo.

Abaixo definimos duas ferramentas de consulta de voos:
- **Primária** — cobre Paris, Tóquio e Barcelona
- **Reserva** — cobre Berlim, Sydney e Cidade de Nova Iorque


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agente Auto-reflexivo com Recuperação de Erros

O agente abaixo é instruído para tentar primeiro o sistema de voo primário, reconhecer falhas e recorrer de forma transparente ao sistema de reserva. Depois de cada resposta, avalia brevemente a si próprio se respondeu completamente à pergunta do utilizador.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Padrão de Autoavaliação

Outra faceta da metacognição é a **autoavaliação**: um agente separado (ou o mesmo agente numa segunda passagem) revê uma resposta quanto à completude, precisão e utilidade.

Abaixo criamos um agente `ResponseEvaluator` que pontua respostas de agentes de viagens em três dimensões.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Resumo

Nesta lição aprendeu a construir **agentes metacognitivos** usando o Microsoft Agent Framework:

- **Autorreflexão**: Agentes que monitorizam o próprio raciocínio e comunicam de forma transparente o que aconteceu.
- **Recuperação de erros com fallbacks**: Um padrão de ferramenta primária + de reserva onde o agente deteta falhas (por exemplo, erros 404) e tenta automaticamente uma fonte alternativa.
- **Autoavaliação**: Um agente avaliador separado que avalia as respostas quanto à completude, precisão e utilidade.

Estes padrões tornam os agentes mais robustos, transparentes e fiáveis — qualidades críticas para implantações de produção.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Isenção de responsabilidade:
Este documento foi traduzido usando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos por garantir a precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original, na sua língua de origem, deve ser considerado a fonte autoritativa. Para informações críticas, recomenda-se uma tradução profissional realizada por um tradutor humano. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
